# 01 — GFS Data Loading with noawclg

This notebook covers the first step in any weather plot: **downloading and inspecting GFS data** using the `noawclg` companion library.

You do **not** need `noaawc` here — just raw data tools.

---
**Stack used:** `noawclg`, `xarray`, `numpy`, `matplotlib`

## 1. Install / import

In [ ]:
# pip install noawclg xarray numpy matplotlib
import noawclg
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

## 2. Load a single GFS variable

`noawclg.load()` downloads GRIB2 files from NOAA servers and returns an `xarray.Dataset`.

| Parameter | Description |
|---|---|
| `var` | Variable key (e.g. `"t2m"`, `"prmsl"`, `"prate"`) |
| `run_date` | Forecast run date (`"YYYYMMDD"` or `"DD/MM/YYYY"`) |
| `cycle` | Run cycle — `"00"`, `"06"`, `"12"`, or `"18"` |
| `forecast_hours` | Iterable of forecast lead times in hours |

In [ ]:
ds = noawclg.load(
    var="t2m",
    run_date="20260418",
    cycle="00",
    forecast_hours=range(0, 25, 3),  # 0, 3, 6, ..., 24 h
)

print(ds)

## 3. Inspect the dataset

In [ ]:
# Coordinates
print("Latitude  :", ds["lat"].values[:5], "...", ds["lat"].values[-5:])
print("Longitude :", ds["lon"].values[:5], "...", ds["lon"].values[-5:])
print("Time steps:", ds["t2m"].time.values)

In [ ]:
# The variable data array
da = ds["t2m"]
print("Shape :", da.shape)   # (time, lat, lon)
print("Dtype :", da.dtype)
print("Units :", da.attrs.get("units", "check noawclg docs"))
print("Min/Max:", float(da.min()), "/", float(da.max()))

### Unit conversion

GFS 2m temperature is in **Kelvin**. Convert to °C:

In [ ]:
# Select the first time step and convert K → °C
lat  = ds["lat"].values          # shape (721,)
lon  = ds["lon"].values          # shape (1440,)  — 0…360
field_k = ds["t2m"].isel(time=0).values   # shape (721, 1440)
field_c = field_k - 273.15

print(f"T2m range: {field_c.min():.1f} … {field_c.max():.1f} °C")

## 4. Quick sanity-check plot (no cartopy yet)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.pcolormesh(lon, lat, field_c, cmap="RdBu_r", vmin=-40, vmax=45)
fig.colorbar(im, ax=ax, label="2m Temperature (°C)")
ax.set_title("GFS 2m Temperature — sanity check (no projection)")
ax.set_xlabel("Longitude (0–360)")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## 5. Longitude convention: 0–360 → −180/180

Cartopy projections expect **−180/180** longitudes. Here is the canonical conversion:

In [ ]:
# 1. Wrap values
lon_180 = np.where(lon > 180, lon - 360, lon)   # still unsorted

# 2. Get the sorted index and reorder both lon and the field columns
order   = np.argsort(lon_180)
lon_180 = lon_180[order]
field_180 = field_c[:, order]

print("Lon range:", lon_180[0], "…", lon_180[-1])

## 6. Load multiple variables at once

In [ ]:
ds_prmsl = noawclg.load(
    var="prmsl",
    run_date="20260418",
    cycle="00",
    forecast_hours=[0],
)

# MSLP is in Pa — convert to hPa
prmsl_hpa = ds_prmsl["prmsl"].isel(time=0).values / 100.0
print(f"MSLP range: {prmsl_hpa.min():.1f} … {prmsl_hpa.max():.1f} hPa")

## 7. Common GFS variable keys

| Key | Description | Typical units |
|---|---|---|
| `t2m` | 2m temperature | K (→ °C: −273.15) |
| `d2m` | 2m dewpoint | K |
| `prmsl` | Mean sea-level pressure | Pa (→ hPa: ÷100) |
| `prate` | Precipitation rate | kg m⁻² s⁻¹ |
| `r2` | 2m relative humidity | % |
| `sh2` | 2m specific humidity | kg/kg |
| `u10` | 10m U-wind | m/s |
| `v10` | 10m V-wind | m/s |
| `pwat` | Precipitable water | kg/m² |
| `cape` | CAPE | J/kg |
| `tcc` | Total cloud cover | fraction |

---
**Next:** [02_orthographic_globe.ipynb](02_orthographic_globe.ipynb)